# Geospatial Machine Learning: Instance Segmentation

This notebook demonstrates how to train an **instance segmentation** model for detecting individual objects (e.g., buildings) in satellite or aerial imagery. We use the [GeoAI](https://opengeoai.org/) library and Mask R-CNN, following the workflow from the [GeoAI instance segmentation example](https://opengeoai.org/examples/train_instance_segmentation_model/).

**Note**: Training a neural network is computationally expensive. Expect several minutes to hours depending on your hardware. You can reduce `num_epochs` for a quicker (but less accurate) run.

## What Problem Does Instance Segmentation Solve?

In geospatial ML, we often need to detect and delineate individual objects—buildings, cars, solar panels—from imagery.

- **Semantic segmentation**: Labels every pixel with a class (e.g., "building" vs "not building") but does not distinguish between separate buildings. All buildings are one blob.
- **Instance segmentation**: Identifies each object separately. Building A gets its own mask, Building B gets another. This lets you count objects, measure each one's area, and analyze spatial relationships.

**Mask R-CNN** combines object detection (bounding boxes) with pixel-level masks, giving you both the location and the precise shape of each instance.

In [ ]:
# Optional: Install GeoAI and dependencies
# %pip install geoai-py

import geoai
from pathlib import Path

## Download Sample Data

We use NAIP (National Agriculture Imagery Program) aerial imagery and building footprint labels from a public Hugging Face dataset. The data includes:
- **Training raster**: RGB aerial image for training
- **Training vectors**: Building polygons (GeoJSON)
- **Test raster**: A separate image for inference

In [ ]:
train_raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_rgb_train.tif"
)
train_vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"
test_raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_test.tif"
)

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

train_raster_path = geoai.download_file(train_raster_url)
train_vector_path = geoai.download_file(train_vector_url)
test_raster_path = geoai.download_file(test_raster_url)

print("Downloaded:", train_raster_path, train_vector_path, test_raster_path)

## Visualize Sample Data

Inspect the training raster metadata and optionally view the imagery and labels interactively.

In [ ]:
geoai.get_raster_info(train_raster_path)

In [ ]:
# Optional: Interactive map of training data with building outlines
# Uncomment to view in Jupyter (requires ipyleaflet)
# style_dict = {"color": "#ff0000", "weight": 2, "opacity": 1, "fillOpacity": 0}
# geoai.view_vector_interactive(
#     train_vector_path, tiles=train_raster_path,
#     style_function=lambda x: style_dict
# )

## Create Training Data

The model needs image tiles and corresponding label masks. `export_geotiff_tiles` cuts the raster into fixed-size tiles (512×512) and rasterizes the vector labels into matching mask images. The `stride` controls overlap between tiles.

In [ ]:
out_folder = str(data_dir / "buildings_instance")
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_vector_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)
print(f"Created {len(tiles)} tiles in {out_folder}")

## Train the Instance Segmentation Model

We train a **Mask R-CNN** model. Key parameters:
- `num_classes=2`: background + building
- `batch_size=4`: Smaller batches due to model complexity (adjust if you have more GPU memory)
- `num_epochs=10`: Reduce for faster runs, increase for better accuracy
- `val_split=0.2`: 20% of tiles held out for validation

**Training can take 10–60+ minutes** depending on hardware.

In [ ]:
geoai.train_instance_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/instance_models",
    num_classes=2,  # background + building
    num_channels=3,
    batch_size=4,
    num_epochs=10,
    learning_rate=0.005,
    val_split=0.2,
    visualize=True,
    verbose=True,
)

## Run Inference

Apply the trained model to the test image. `instance_segmentation` uses a sliding window to process large rasters. Each window is classified, and overlapping predictions are merged.

In [ ]:
masks_path = str(data_dir / "naip_test_instance_prediction.tif")
model_path = f"{out_folder}/instance_models/best_model.pth"

geoai.instance_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    num_classes=2,
    num_channels=3,
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
)

## Vectorize Masks

The model outputs a raster mask (each pixel = instance ID). For GIS workflows, we convert these to vector polygons using `orthogonalize`, which also regularizes building outlines.

In [ ]:
output_vector_path = str(data_dir / "naip_test_instance_prediction.geojson")
gdf = geoai.orthogonalize(masks_path, output_vector_path, epsilon=2)
print(f"Detected {len(gdf)} buildings")

## Add Geometric Properties

Compute area, perimeter, and other properties for each detected building. Useful for filtering (e.g., remove tiny artifacts) and analysis.

In [ ]:
gdf_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")
print(gdf_props[["area_m2", "perimeter_m"]].head())

## Visualize and Filter Results

Visualize the predicted masks and vectors. Filtering by area (e.g., > 50 m²) removes small false positives.

In [ ]:
# Optional: Interactive visualization (uncomment if ipyleaflet is available)
# geoai.view_raster(masks_path, nodata=0, cmap="tab20", basemap=test_raster_path, backend="ipyleaflet")
# geoai.view_vector_interactive(gdf_props, column="area_m2", tiles=test_raster_path)

In [ ]:
gdf_filtered = gdf_props[gdf_props["area_m2"] > 50]
print(f"After filtering (area > 50 m²): {len(gdf_filtered)} buildings")

## Instance vs Semantic Segmentation: When to Use Each

| Use Instance Segmentation when… | Use Semantic Segmentation when… |
|--------------------------------|----------------------------------|
| You need to count or measure individual objects | You only need area coverage or pixel-level class maps |
| Objects may overlap or touch | Features are continuous (e.g., vegetation, water) |
| You need spatial relationships between objects | You want faster training and inference |
| Examples: buildings, cars, solar panels | Examples: land cover, flood extent |

**Model outputs**: Instance segmentation gives bounding boxes, confidence scores, and a separate mask per object. Semantic segmentation gives one class per pixel with no object boundaries.

## Optional: Plot Training Metrics

Examine loss and validation metrics to assess training quality.

In [ ]:
# Uncomment to plot training curves
# geoai.plot_performance_metrics(
#     history_path=f"{out_folder}/instance_models/training_history.pth",
#     figsize=(15, 5),
#     verbose=True,
# )